# C5 — Metadata Ablation Report

**Protocol:** `v1.0.0`  
**Protocol hash:** `d42337690181f1054297f514934ad0c98bb718223bc06d8de5569f40a184ee32`

Notebook ini adalah presentation layer read-only untuk analisis sekunder/eksploratif feature set A–D. Seluruh angka diregenerasi dari OOF prediction canonical; tidak ada fitting, tuning, atau akses official NIH test.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'run_experiment.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.protocol.stages import load_model_lock, oof_path_for
from src.reporting.canonical import load_canonical_context, load_oof_predictions, metric_table

context = load_canonical_context(PROJECT_ROOT)
protocol_dir = context.protocol_dir
lock = load_model_lock(protocol_dir)
assert (protocol_dir / 'ablation' / '_SUCCESS').is_file(), 'C5 belum lengkap'
print('protocol_hash:', context.protocol_hash)
print('backbone lock:', lock['selected_backbone'], lock['selected_pretraining'])

## Muat dan audit OOF A–D
Feature set D menggunakan artefak C4; A/B/C menggunakan artefak C5.

In [ ]:
models = {}
for scenario in ('S1', 'S3'):
    for feature_set in ('A', 'B', 'C', 'D'):
        stage = 'C4' if feature_set == 'D' else 'C5'
        stage_dir = 'main' if stage == 'C4' else 'ablation'
        model = 'canonical_mlp' if scenario == 'S1' else lock['selected_backbone']
        pretraining = 'not_applicable' if scenario == 'S1' else lock['selected_pretraining']
        path = oof_path_for(protocol_dir, stage=stage, scenario=scenario, model=model, pretraining=pretraining, feature_set=feature_set)
        models[f'{scenario}-{feature_set}'] = load_oof_predictions(context, path, expected_stage_directory=stage_dir)

ablation_metrics = metric_table(models).reset_index()
ablation_metrics[['model', 'roc_auc_pooled', 'roc_auc_mean', 'roc_auc_sd', 'ap_pooled', 'brier_score']]

In [ ]:
plot_data = ablation_metrics[ablation_metrics['model'].str.startswith('S3-')].copy()
plot_data['feature_set'] = plot_data['model'].str[-1]
plot_data = plot_data.sort_values('feature_set')
ax = plot_data.plot.bar(x='feature_set', y='roc_auc_pooled', ylim=(0.74, 0.76), legend=False, figsize=(7, 4))
ax.set(title='S3 OOF ROC-AUC menurut feature set', xlabel='Feature set', ylabel='Pooled OOF ROC-AUC')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()

## Batas interpretasi
Selisih A–D dilaporkan sebagai *incremental predictive contribution under the tested feature configuration*, bukan kontribusi kausal. Selang kepercayaan berpasangan yang menjadi rujukan resmi dibaca dari notebook C6.